In [46]:
import pandas as pd
import numpy as np

from matplotlib import pyplot
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report,confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot as plt
from xgboost import XGBClassifier

In [47]:
df = pd.read_csv ('data/Customer_Loan_Request.csv')
#data = pd.read_csv ('data/Customer_Loan_Request.csv')

df.head (3)   

,CustomerID,Name,Age,Gender,MaritalStatus,EducationLevel,EmploymentStatus,AnnualIncome,LoanAmountRequested,PurposeOfLoan,CreditScore,ExistingLoansCount,LatePaymentsLastYear,LoanApproved
0,6b65a6a4-8b81-48f6-b38a-088ca65ed389,Carla Briggs,56,Male,Widowed,PhD,Employed,175074,30737,Home,557,0,6,No
1,5be6128e-18c2-4797-a142-ea7d17be3111,William Lewis,53,Male,Widowed,Other,Unemployed,100000,36560,Car,722,4,6,No
2,7412b293-4729-4739-a14f-f3d719db3ad0,Rose Jackson,19,Male,Married,Bachelor,Unemployed,220000,2264,Car,790,2,2,No


In [48]:
# ============================================================
# 2. FEATURES
# ============================================================

features = [
    'MaritalStatus',
    'EducationLevel',
    'EmploymentStatus',
    'AnnualIncome',
    'LoanAmountRequested',
    'ExistingLoansCount',
    'LatePaymentsLastYear',
    'CreditScore'
]

X = df[features]

X = df.drop(columns=["LoanApproved"])

df["LoanApproved"] = df["LoanApproved"].replace({"No": 0, "Yes": 1})

y = df["LoanApproved"]

# ============================================================
# 3. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)


C:\Users\khalid.ahmed\AppData\Local\Temp\ipykernel_18188\299037263.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["LoanApproved"] = df["LoanApproved"].replace({"No": 0, "Yes": 1})


In [49]:
# ============================================================
# 4. PREPROCESSING
# ============================================================

# Excel Sheet Columns

#Numeric Fields
# Age
# AnnualIncome
# LoanAmountRequest
# CreditScore
# ExistingLoansCounts
# LatePaymentLastYears
# LoanApproved

# Categorical Fields
# CustomerID
# Name
# Gender
# MartialStatus
# EducationLevel
# EmploymentStatus
# PurposeofLoan

numeric_features = [
    'AnnualIncome',
    'LoanAmountRequested',
    'CreditScore',
    'ExistingLoansCount',
    'LatePaymentsLastYear'
]

categorical_features = [
    'MaritalStatus',
    'EducationLevel',
    'EmploymentStatus'
]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


In [52]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=3,
        random_state=42
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

df = X_test.copy()   # or your test dataframe

df["y_prob"] = y_prob
df["y_pred"] = (df["y_prob"] > 0.65).astype(int)

pd.DataFrame ({"y_test":y_test, "y_pred": y_pred, "y_prob":y_prob})


,y_test,y_pred,y_prob
7212,1,1,0.705183
512,0,1,0.685592
586,0,1,0.696223
6433,1,1,0.697686
6243,1,1,0.693426
...,...,...,...
8998,1,1,0.705200
2442,0,1,0.697591
3061,1,1,0.699052
6189,1,1,0.681574


In [57]:

# Updated Test

numeric_features = [
    'Age',
    'AnnualIncome',
    'LoanAmountRequested',
    'CreditScore',
    'ExistingLoansCount',
    'LatePaymentsLastYear'
]

categorical_features = [
    'Gender',
    'MaritalStatus',
    'EducationLevel',
    'EmploymentStatus',
    'PurposeOfLoan'
]

RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

df["Prediction_Default"] = y_pred
df["Prediction_065"] = (y_prob > 0.65).astype(int)

from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

model = pipeline.named_steps["model"]

print(model.feature_importances_)

[[   0  539]
 [   0 1261]]
[0.20214406 0.19780842 0.17094826 0.0607221  0.08145442 0.03701745
 0.01421963 0.02877223 0.01946159 0.01186488 0.03093442 0.01062174
 0.01787224 0.01503135 0.02040499 0.0159558  0.0191789  0.02927776
 0.01630973]


In [62]:
print(df.columns.tolist())

['CustomerID', 'Name', 'Age', 'Gender', 'MaritalStatus', 'EducationLevel', 'EmploymentStatus', 'AnnualIncome', 'LoanAmountRequested', 'PurposeOfLoan', 'CreditScore', 'ExistingLoansCount', 'LatePaymentsLastYear']


In [54]:
print(y_train.value_counts())
print(y_test.value_counts())

print(df["y_prob"].describe())

LoanApproved
1    5042
0    2158
Name: count, dtype: int64
LoanApproved
1    1261
0     539
Name: count, dtype: int64
count    1800.000000
mean        0.699506
std         0.010415
min         0.652419
25%         0.693287
50%         0.700643
75%         0.707137
max         0.720256
Name: y_prob, dtype: float64


In [33]:
print(y_prob[:20])
print("Minimum:", y_prob.min())
print("Maximum:", y_prob.max())

[0.70738323 0.68649079 0.70073549 0.70064827 0.69512648 0.70496357
 0.70836566 0.69920723 0.6952858  0.70683574 0.69772162 0.707925
 0.68822782 0.69497607 0.69319644 0.71397217 0.71295482 0.6990767
 0.69667722 0.70177919]
Minimum: 0.6400263861681823
Maximum: 0.7277791822523763


In [34]:

# ============================================================
# 9. RISK FUNCTIONS
# ============================================================

def risk_grade(prob):
    if prob < 0.25:
        return "A"
    elif 0.25 <= prob <= 0.45:
        return "B"
    elif 0.45 < prob <= 0.65:
        return "C"
    elif 0.65 < prob <= 0.85:
        return "D"
    else:
        return "E"


def risk_statement(prob):
    if prob < 0.25:
        return "Very Low Risk"
    elif 0.25 <= prob <= 0.45:
        return "Low Risk"
    elif 0.45 < prob <= 0.65:
        return "Medium Risk"
    elif 0.65 < prob <= 0.85:
        return "High Risk"
    else:
        return "Very High Risk"


def loan_decision(prob):
    if prob < 0.45:
        return "Approved"
    elif prob < 0.65:
        return "Manual Review"
    else:
        return "Rejected"
        

In [35]:
# ============================================================
# 10. CREDIT SCORE (300-900)
# ============================================================

credit_score = (900 - (y_prob * 600)).astype(int)

# ============================================================
# 11. DETAILED CUSTOMER REPORT
# ============================================================

df_result = X_test.copy()

df_result["DefaultProbability"] = y_prob.round(4)

df_result["CreditScoreGenerated"] = credit_score

df_result["RiskGrade"] = df_result["DefaultProbability"].apply(
    risk_grade
)

df_result["RiskStatement"] = df_result["DefaultProbability"].apply(
    risk_statement
)

df_result["Decision"] = df_result["DefaultProbability"].apply(
    loan_decision
)

print("\nDetailed Results")
print(df_result.head())



Detailed Results
                                CustomerID             Name  Age  Gender  \
7212  fdf48930-a265-4f65-8d56-e00031456a6c       Luke Smith   46  Female   
512   47198f5c-f69b-4596-bf10-938caec7f267     Tracey White   63    Male   
586   a19f2485-31c4-4c9e-b0be-96304a6333e3     Sandra Lopez   38    Male   
6433  da2cf852-0a19-4939-9fc7-ab262cb941fc      Robert Lane   31  Female   
6243  57d337d3-4fd8-4bbb-992c-11072444d301  Nicole Roberson   50  Female   

     MaritalStatus EducationLevel EmploymentStatus  AnnualIncome  \
7212       Married          Other          Retired        137211   
512         Single       Bachelor       Unemployed         21942   
586       Divorced          Other          Retired        186429   
6433      Divorced          Other          Retired        109652   
6243      Divorced            PhD    Self-employed        175198   

      LoanAmountRequested PurposeOfLoan  CreditScore  ExistingLoansCount  \
7212                30897          Home 

In [36]:

# ============================================================
# 12. MANAGEMENT SUMMARY
# ============================================================

summary = (
    df_result
    .groupby(
        ["RiskGrade",
         "RiskStatement",
         "Decision"]
    )
    .size()
    .reset_index(name="CustomerCount")
)

print("\nManagement Summary")
print(summary)

# ============================================================
# 13. DECISION SUMMARY
# ============================================================

print("\nDecision Summary\n")
print(df_result["Decision"].value_counts())

# ============================================================
# 14. RISK SUMMARY
# ============================================================

print("\nRisk Summary\n")
print(df_result["RiskStatement"].value_counts())



Management Summary
  RiskGrade RiskStatement       Decision  CustomerCount
0         C   Medium Risk  Manual Review              2
1         D     High Risk       Rejected           1798

Decision Summary

Decision
Rejected         1798
Manual Review       2
Name: count, dtype: int64

Risk Summary

RiskStatement
High Risk      1798
Medium Risk       2
Name: count, dtype: int64


In [37]:
risk_grade_summary = (
    df_result
    .groupby("RiskGrade")
    .size()
    .reset_index(name="CustomerCount")
)

risk_grade_summary["Percentage"] = (
    risk_grade_summary["CustomerCount"] / risk_grade_summary["CustomerCount"].sum() * 100
)

print(risk_grade_summary)

  RiskGrade  CustomerCount  Percentage
0         C              2    0.111111
1         D           1798   99.888889


In [39]:
# ============================================================
# 15. FEATURE IMPORTANCE
# ============================================================

feature_names = (
    pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = (
    pipeline.named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
)

#print("\nTop Important Features")
#print(importance_df.head(20))


In [40]:
# ============================================================
# 16. EXPORT RESULTS
# ============================================================

# df_result.to_excel(
#     "Loan_Risk_Assessment.xlsx",
#     index=False
# )



# def Loan_Risk_Summary_Report():

summary.to_excel("Loan_Risk_Summary.xlsx", index=False)

print("\nFiles Exported Successfully")


# Loan_Risk_Summary_Report.py

# %%writefile test.py
# def Loan_Risk_Summary_Report():

#     summary.to_excel(

#         "Loan_Risk_Summary.xlsx",
#         index=False
#     )

#    print("Files Exported Successfully")



Files Exported Successfully


In [15]:

# with open("Loan_Risk_Summary_Report.py", "w") as f:
#     f.write("""
# def Loan_Risk_Summary_Report(summary):
#     summary.to_excel("Loan_Risk_Summary.xlsx", index=False)
#     print("Files Exported Successfully")
# """)


In [42]:
def make_risk_statement(row):
    dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)
    
    # Hard risk signals first
    if row["CreditScore"] < 600 or row["LatePaymentsLastYear"] > 2:
        return "High risk due to weak credit behavior"
    
    # Financial burden
    elif dti > 3 or row["ExistingLoansCount"] > 3:
        return "Moderate risk due to high debt burden"
    
    # Soft signals (employment context)
    elif row["EmploymentStatus"] in ["Unemployed", "Student"]:
        return "Moderate risk due to employment status"
    
    else:
        return "Low risk based on current customer profile"


def make_decision(row):
    dti = row["LoanAmountRequested"] / (row["AnnualIncome"] + 1)

    # Hard reject
    if row["CreditScore"] < 600 or row["LatePaymentsLastYear"] > 2:
        return "Reject"

    # Impossible repayment
    if dti > 5:
        return "Reject"

    # Manual review zone
    if (dti > 3 or
        row["ExistingLoansCount"] >= 1 or        # ← fix >= 3
        row["CreditScore"] < 650 or              # ← borderline buffer
        row["EmploymentStatus"] in ["Unemployed", "Student", "Retired"]):  # ← Retired added
        return "Review manually"

    return "Approve"


In [43]:
print(type(y_test))
print(len(y_test))
print(y_test.shape)

<class 'pandas.core.series.Series'>
1800
(1800,)


In [44]:
print(y_test.head())

7212    1
512     0
586     0
6433    1
6243    1
Name: LoanApproved, dtype: int64


In [45]:
df = X_test.copy()

df["LoanApproved"] = y_test.values
df["y_prob"] = y_prob
df["y_pred"] = y_pred

df["LoanApproved"] = df["LoanApproved"].replace({0: "No", 1: "Yes"})
df["Prediction"] = df["y_pred"].replace({0: "No", 1: "Yes"})

#df["LoanApproved"] = df["LoanApproved"].replace({0: "No", 1: "Yes"})
#df = df.drop(columns=["LoanApprovedLabel"], errors="ignore")

# First calculate probabilities
df["y_prob"] = pipeline.predict_proba(X)[:, 1]

# Then calculate predictions
df["y_pred"] = (df["y_prob"] > 0.5).astype(int)

df["riskstatement"] = df.apply(make_risk_statement, axis=1)
df["decision"] = df.apply(make_decision, axis=1)

cols = df.columns.tolist()

# Swap y_prob and y_pred
df["y_pred"] = df["y_pred"].replace({0: "No", 1: "Yes"})
df["y_prob"] = pd.to_numeric(df["y_prob"], errors="coerce")
df["y_prob"] = (df["y_prob"] * 100).round(2)

i = cols.index("y_prob")
j = cols.index("y_pred")
cols[i], cols[j] = cols[j], cols[i]

# Reorder DataFrame
df = df[cols]

df = df.loc[:, ~df.columns.duplicated()]

df.rename(columns={
    "y_pred": "Prediction",
    "y_prob": "Probability"
}, inplace=True)

# Export
df.to_excel("Loan_Risk_Assessment_AI_BusinessLogic.xlsx", index=False)

print("\nFiles Exported Successfully")

ValueError: Length of values (9000) does not match length of index (1800)

In [17]:
print (df)

                                CustomerID            Name  Age  Gender  \
0     6b65a6a4-8b81-48f6-b38a-088ca65ed389    Carla Briggs   56    Male   
1     5be6128e-18c2-4797-a142-ea7d17be3111   William Lewis   53    Male   
2     7412b293-4729-4739-a14f-f3d719db3ad0    Rose Jackson   19    Male   
3     a28defe3-9bf0-4273-9247-6f57a5e5a5ab  Rebecca Lowery   39    Male   
4     b02b61c4-a3d7-4628-ace6-6fa2fd5166e6       Amy Olsen   66    Male   
...                                    ...             ...  ...     ...   
8995  4d1f3b9a-04ca-4bf5-9b97-9060a7f12456   Maria Barrera   61  Female   
8996  b6d62dfd-7365-4790-a9cd-17503b63be00   Raymond Watts   27  Female   
8997  bdcfc42b-8122-41db-aa6a-58b294568b74     Jose Torres   22  Female   
8998  a874d557-cd0e-4708-acbf-81d4b1720b65  Michael Butler   26  Female   
8999  f3151b0a-fd13-4ac5-a6cf-af54ca9dc28c   Jackie Carter   55  Female   

     MaritalStatus EducationLevel EmploymentStatus  AnnualIncome  \
0          Widowed            P

In [18]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, classification_report
print('Accuracy Score: ', accuracy_score(y_test, y_pred))
print('Recall Score: ', recall_score(y_test, y_pred))
print('Precision Score: ', precision_score(y_test, y_pred))
print('F1 Score: ', f1_score(y_test, y_pred))
print('Confusion Matrix: \n', confusion_matrix(y_test, y_pred))
print('Classification Report: \n', classification_report(y_test, y_pred))

Accuracy Score:  0.7005555555555556
Recall Score:  1.0
Precision Score:  0.7005555555555556
F1 Score:  0.8239137536752695
Confusion Matrix: 
 [[   0  539]
 [   0 1261]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       539
           1       0.70      1.00      0.82      1261

    accuracy                           0.70      1800
   macro avg       0.35      0.50      0.41      1800
weighted avg       0.49      0.70      0.58      1800



C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [208]:
pip install joblib

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [138]:
import joblib

In [209]:
joblib.dump(pipeline, "Loan_Risk_Model.pkl")

['Loan_Risk_Model.pkl']

In [210]:
import os

print(os.path.exists("Loan_Risk_Model.pkl"))

True
